# Efficient attention in DreamSim

## Imports

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/vmazashvili/efficient-attention-dreamsim.git'
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path('/content/efficient-attention-dreamsim')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(['git', '-C', str(ROOT), 'pull', '-q'], check=True)
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'configs' / 'base.yaml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

import datetime
import shutil

import torch
from torch.utils.data import DataLoader

from effsim.backbone import embed, load_dino
from effsim.data import TripletDataset, extract_shards, load_triplets
from effsim.evaluate import append_run, save_embeddings, score_2afc, summarise

## Globals

In [ ]:
import yaml
CFG = yaml.safe_load(open(ROOT / 'configs' / 'base.yaml'))
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_ROOT = Path(CFG['data_root'])
DRIVE_DATA = Path('/content/drive/MyDrive/nights224')
DRIVE_OUT = Path('/content/drive/MyDrive/effsim')
QUICK = True  # True: smoke val only, nothing logged. False: full val + test, rows in results/runs.csv
BATCH = 64
print(f'colab={IN_COLAB} root={ROOT} device={DEVICE} quick={QUICK}')

## Utils

## Data

In [ ]:
from collections import Counter

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not (DATA_ROOT / 'data.csv').exists():
        print(extract_shards(DRIVE_DATA, DATA_ROOT), 'shards extracted to', DATA_ROOT)

CSV = DATA_ROOT / 'data.csv'
SPLITS = {'val': load_triplets(CSV, 'val', ROOT / 'data/splits/smoke_val.csv' if QUICK else None)}
if not QUICK:
    SPLITS['test'] = load_triplets(CSV, 'test')
print({k: len(v) for k, v in SPLITS.items()})
test_imagenet = Counter(r['is_imagenet'].upper() == 'TRUE' for r in load_triplets(CSV, 'test'))
print(f"test is_imagenet: {test_imagenet[True]} / {test_imagenet[False]} (expected 1041 / 783)")


def loader(split, normalize):
    ds = TripletDataset(DATA_ROOT, SPLITS[split], normalize=normalize)
    return ds, DataLoader(ds, batch_size=BATCH, num_workers=2, pin_memory=DEVICE == 'cuda')

## Network

In [ ]:
dino = load_dino().to(DEVICE)
MODELS = {  # run_id -> (stage, embed_fn, ImageNet-normalised input?)
    'B0': ('zero-shot', lambda x: embed(dino, x), True),
}
if IN_COLAB or DEVICE == 'cuda':
    from effsim.brel import load_brel
    for run_id, norm in (('B-rel', False), ('B-rel-norm', True)):
        MODELS[run_id] = ('release', load_brel(DEVICE, DRIVE_OUT / 'dreamsim_cache', normalize_embeds=norm), False)

## Train

## Evaluation

In [ ]:
commit = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip()
gpu = torch.cuda.get_device_name() if DEVICE == 'cuda' else 'cpu'
for run_id, (stage, embed_fn, normalize) in MODELS.items():
    scores = {}
    for split in SPLITS:
        ds, dl = loader(split, normalize)
        res = score_2afc(embed_fn, dl, DEVICE, keep_embeddings=split == 'test')
        scores[split] = summarise(res, ds.is_imagenet)
        print(run_id, split, {k: round(v, 4) if isinstance(v, float) else v for k, v in scores[split].items()})
        if split == 'test':
            save_embeddings(DRIVE_OUT / 'embeddings' / f'{run_id}_test', res)
    if not QUICK:
        append_run(ROOT / 'results' / 'runs.csv', run_id=run_id, stage=stage, epochs=0,
                   val_2afc=scores['val']['all'], test_2afc=scores['test']['all'],
                   test_2afc_imagenet=scores['test']['imagenet'],
                   test_2afc_no_imagenet=scores['test']['no_imagenet'],
                   gpu=gpu, commit=commit, date=datetime.date.today().isoformat())

if not QUICK and IN_COLAB:
    shutil.copyfile(ROOT / 'results' / 'runs.csv', DRIVE_OUT / 'runs.csv')
    print(open(ROOT / 'results' / 'runs.csv').read())